# StrongReject Result Compilation

This notebook compiles the workflow-traced StrongReject experiment caches and displays the main summary, coverage, and reliability tables.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt

EXPECTED_PYTHON = Path("/workspace/.venv/bin/python")
if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Wrong notebook kernel.\n"
        f"  got:      {sys.executable}\n"
        f"  expected: {EXPECTED_PYTHON}\n"
        "In Cursor: Select Kernel -> Python Environments -> Enter interpreter path:\n"
        "  /workspace/.venv/bin/python\n"
        "Or run from terminal: bash results/run_compile_notebook.sh"
    )

def _resolve_extension_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'compile_strongreject_results.py').exists():
            return candidate
        nested = candidate / 'activation_oracles_extensions' / 'compile_strongreject_results.py'
        if nested.exists():
            return nested.parent
    raise FileNotFoundError('Could not locate compile_strongreject_results.py from current notebook cwd')

EXT_ROOT = _resolve_extension_root(Path.cwd())
os.chdir(EXT_ROOT)
sys.path.insert(0, str(EXT_ROOT))
sys.path.insert(0, str(EXT_ROOT / 'results'))

from compile_strongreject_results import StrongRejectCompileConfig, compile_strongreject_results
from viz_helpers import (
    PathAliaser,
    apply_display_transforms,
    apply_probe_sort,
    percent_style,
    plot_summary,
    probe_order_map,
    rename_display_columns,
)
from result_validation_helpers import (
    apply_filter,
    build_coverage_df,
    build_peek_table,
    display_coverage_report,
)

## Configure And Compile

In [ ]:
cfg = StrongRejectCompileConfig(
    cache_root=EXT_ROOT / 'cache',
    output_dir=EXT_ROOT / 'results' / 'compiled_strongreject_results',
    judge_instruction_path='strongReject_v5.jinja2',
    expected_target_prompts=100,
    expected_target_rollouts=50,
    expected_oracle_rollouts=50,
)

manifest = compile_strongreject_results(cfg)
manifest['outputs']

## Coverage

In [3]:
from result_validation_helpers import build_coverage_df
build_coverage_df(manifest).style.format({'coverage_pct': '{:.1%}'})

,condition,expected_files,loaded_files,missing_files,coverage_pct
0,oracle_rollout_control,100,100,0,100.0%
1,target_baseline,100,100,0,100.0%
2,target_rollout_oracle,200,200,0,100.0%
3,user_prompt_oracle,200,200,0,100.0%


In [4]:
from result_validation_helpers import display_coverage_report
display_coverage_report(manifest, cfg)

Missing expected files: 0
Malformed files: 0
Skipped score leaves (probe entries with no accepted numeric StrongReject score): 0


,condition,target_prompt_index,oracle_prompt_file,probe_kind,probe_name,expected_scored,actual_scored
0,user_prompt_oracle,90,prompts/oracle_prompts/default_oracle_prompts....,token_points,last_prompt_token,50,49
1,user_prompt_oracle,10,prompts/oracle_prompts/default_oracle_prompts....,token_points,token_before_im_end,50,49
2,user_prompt_oracle,53,prompts/oracle_prompts/default_oracle_prompts....,token_points,token_before_im_end,50,48
3,user_prompt_oracle,68,prompts/oracle_prompts/default_oracle_prompts....,token_points,token_before_im_end,50,49
4,user_prompt_oracle,97,prompts/oracle_prompts/default_oracle_prompts....,token_points,token_before_im_end,50,49
5,user_prompt_oracle,43,prompts/oracle_prompts/default_oracle_prompts....,token_points,trailing_im_start_token,50,49
6,user_prompt_oracle,67,prompts/oracle_prompts/default_oracle_prompts....,token_points,trailing_im_start_token,50,49
7,user_prompt_oracle,23,prompts/oracle_prompts/model_answer_min_200_wo...,full_seq,full_seq,50,49
8,user_prompt_oracle,47,prompts/oracle_prompts/model_answer_min_200_wo...,full_seq,full_seq,50,49
9,user_prompt_oracle,2,prompts/oracle_prompts/model_answer_min_200_wo...,token_points,im_end_token,50,49


""


No missing cache files in current compile pass.
No skipped score leaves.


## Main Summary

In [ ]:
summary = pd.read_csv(cfg.output_dir / 'strongreject_summary.csv')
prompt_level = pd.read_csv(cfg.output_dir / 'strongreject_prompt_level.csv')
reliability = pd.read_csv(cfg.output_dir / 'strongreject_reliability.csv')
details = pd.read_csv(cfg.output_dir / 'strongreject_details.csv')

all_alias_paths = []
if 'cache_path' in details.columns:
    all_alias_paths.extend([str(x) for x in details['cache_path'].dropna().unique().tolist()])
all_alias_paths.extend([str(x) for x in manifest.get('missing_files', [])])

path_aliaser = PathAliaser(cfg.target_model_name, cfg.cache_root, cfg.output_dir, all_alias_paths)
PROBE_ORDER = probe_order_map(details)

within_prompt_se_col = f'SE Within Prompt Across Oracle Rollouts (N={cfg.expected_oracle_rollouts})'
summary[within_prompt_se_col] = summary['mean_within_prompt_sd'] / summary['mean_within_prompt_n'].pow(0.5)

display_cols = [
    'condition', 'probe_name', 'oracle_prompt_file', 'n_prompts',
    'mean_score', 'se_score', within_prompt_se_col, 'asr_0_2', 'asr_0_5', 'asr_0_8', 'asr_1'
]

# Display-only baseline rule: probe name shown as null for baseline rows.
summary_display = summary.copy()
baseline_mask = summary_display['condition'].isin(['target_baseline', 'oracle_rollout_control'])
summary_display.loc[baseline_mask, 'probe_name'] = None

provenance = (
    details.groupby(['condition', 'preset_source', 'oracle_prompt_file', 'probe_kind', 'probe_name'], dropna=False)
    .agg(
        n_rows=('score', 'size'),
        n_target_prompts=('target_prompt_key', 'nunique'),
        n_cache_files=('cache_path', 'nunique'),
        mean_score=('score', 'mean')
    )
    .reset_index()
)
provenance = path_aliaser.add_alias_column(provenance, 'oracle_prompt_file', 'oracle_prompt_file_alias')
provenance = apply_probe_sort(provenance, PROBE_ORDER)
provenance = apply_display_transforms(provenance)

print(f"Detail rows loaded: {len(details):,}")
print(f"Unique cache files referenced: {details['cache_path'].nunique():,}")

summary_main = apply_display_transforms(summary_display[display_cols])
summary_main = apply_probe_sort(summary_main, PROBE_ORDER)
display(percent_style(rename_display_columns(summary_main.head(10))))
display(percent_style(rename_display_columns(provenance[['condition', 'preset_source', 'oracle_prompt_file', 'probe_name', 'n_rows', 'n_target_prompts', 'n_cache_files', 'mean_score']].head(10))))

## Baselines

In [ ]:
baselines = summary_display.loc[summary_display['condition'].isin(['target_baseline', 'oracle_rollout_control']), display_cols]
baselines = apply_display_transforms(baselines)
baselines = apply_probe_sort(baselines, PROBE_ORDER)
display(percent_style(rename_display_columns(baselines)))

## User Prompt Oracle

In [ ]:
user_prompt = summary_display[summary_display['condition'] == 'user_prompt_oracle']
user_prompt = apply_display_transforms(user_prompt)

user_prompt_table = apply_probe_sort(user_prompt[display_cols], PROBE_ORDER)
display(percent_style(rename_display_columns(user_prompt_table)))

sample_paths = (
    details[details['condition'] == 'user_prompt_oracle'][['oracle_prompt_file', 'probe_kind', 'probe_name', 'cache_path']]
    .drop_duplicates()
)
sample_paths = path_aliaser.add_alias_column(sample_paths, 'cache_path', 'cache_path_alias')
sample_paths = apply_probe_sort(sample_paths, PROBE_ORDER)
sample_paths = apply_display_transforms(sample_paths)
display(rename_display_columns(sample_paths[['oracle_prompt_file', 'probe_name', 'cache_path_alias']].head(50)))

## Target Rollout Oracle

In [ ]:
target_rollout = summary_display[summary_display['condition'] == 'target_rollout_oracle']
target_rollout = apply_display_transforms(target_rollout)
target_rollout_table = apply_probe_sort(target_rollout[display_cols], PROBE_ORDER)
display(percent_style(rename_display_columns(target_rollout_table)))

sample_paths = (
    details[details['condition'] == 'target_rollout_oracle'][['oracle_prompt_file', 'probe_name', 'cache_path']]
    .drop_duplicates()
)
sample_paths = path_aliaser.add_alias_column(sample_paths, 'cache_path', 'cache_path_alias')
sample_paths = apply_probe_sort(sample_paths, PROBE_ORDER)
sample_paths = apply_display_transforms(sample_paths)
display(rename_display_columns(sample_paths[['oracle_prompt_file', 'probe_name', 'cache_path_alias']].head(50)))

## Reliability
`mean_within_prompt_n` is the average number of scored rollouts per prompt used when computing within-prompt standard deviation for that row.


In [ ]:
rel = reliability.copy()
rel = rel.drop(columns=['oracle_prompt', 'oracle_prompt_index'], errors='ignore')
rel = apply_probe_sort(rel, PROBE_ORDER)
rel = apply_display_transforms(rel)
rel = apply_probe_sort(rel, PROBE_ORDER)
display(percent_style(rename_display_columns(rel.head(100))))

INSPECT = {
    'condition': 'user_prompt_oracle',
    'oracle_prompt_file': None,
    'probe_kind': None,
    'probe_name': None,
    'target_prompt_index': None,
    'target_rollout_index': None,
    'oracle_rollout_index': None,
    'max_rows': 20,
}

filtered = apply_filter(details, INSPECT).head(INSPECT['max_rows'])
filtered = path_aliaser.add_alias_column(filtered, 'cache_path', 'cache_path_alias')
filtered = apply_probe_sort(filtered, PROBE_ORDER)

view_cols = [
    'condition', 'target_prompt_index', 'rollout_index', 'target_rollout_index',
    'oracle_rollout_index', 'probe_name', 'oracle_prompt_file',
    'score', 'cache_path_alias'
]
display(path_aliaser.legend_df())
display(percent_style(rename_display_columns(apply_display_transforms(filtered[view_cols]))))

peek_df = build_peek_table(filtered, path_aliaser)
display(percent_style(rename_display_columns(peek_df), extra_pct_cols=['score']))

## Quick Plot

In [ ]:
plot_summary(summary, manifest)